# Sixpack_Rips relative 재계산 (C 가속)
Final_Vector/Sixpack_Rips의 relative를 max_eps=10으로 재계산하여 덮어쓰기.
Column reduction을 C로 가속 (~3.7x).

In [ ]:
!pip install gudhi persim
from google.colab import drive
drive.mount('/content/drive')

## 1. C Column Reduction 컴파일

In [ ]:
%%writefile reduce_c.c
#include <stdlib.h>
#include <string.h>

typedef struct { int *data; int len; int cap; } Col;

static void col_init(Col *c, const int *src, int n) {
    c->len = n; c->cap = n > 0 ? n : 1;
    c->data = (int *)malloc(c->cap * sizeof(int));
    if (n > 0) memcpy(c->data, src, n * sizeof(int));
}
static void col_free(Col *c) { free(c->data); }

static void col_xor(Col *a, const Col *b) {
    int na = a->len, nb = b->len;
    int *tmp = (int *)malloc((na + nb) * sizeof(int));
    int i = 0, j = 0, k = 0;
    while (i < na && j < nb) {
        if (a->data[i] < b->data[j]) tmp[k++] = a->data[i++];
        else if (a->data[i] > b->data[j]) tmp[k++] = b->data[j++];
        else { i++; j++; }
    }
    while (i < na) tmp[k++] = a->data[i++];
    while (j < nb) tmp[k++] = b->data[j++];
    free(a->data);
    a->data = tmp; a->len = k; a->cap = na + nb;
}

void reduce_boundary(const int *flat, const int *offsets, int m, int *low) {
    Col *R = (Col *)malloc(m * sizeof(Col));
    int *piv = (int *)malloc(m * sizeof(int));
    memset(piv, -1, m * sizeof(int));
    for (int i = 0; i < m; i++) {
        col_init(&R[i], flat + offsets[i], offsets[i+1] - offsets[i]);
        low[i] = -1;
    }
    for (int i = 0; i < m; i++) {
        while (R[i].len > 0) {
            int li = R[i].data[R[i].len - 1];
            if (piv[li] >= 0) col_xor(&R[i], &R[piv[li]]);
            else { piv[li] = i; low[i] = li; break; }
        }
    }
    for (int i = 0; i < m; i++) col_free(&R[i]);
    free(R); free(piv);
}

In [ ]:
!gcc -O3 -shared -fPIC -o reduce_c.so reduce_c.c
print('✓ reduce_c.so compiled')

## 2. 함수 정의

In [ ]:
import os, gc, time, ctypes, numpy as np, psutil
from gudhi import RipsComplex
from persim import PersistenceImager
import persim.images_weights as weights
from collections import defaultdict

# C library
lib = ctypes.CDLL('./reduce_c.so')
lib.reduce_boundary.argtypes = [
    ctypes.POINTER(ctypes.c_int), ctypes.POINTER(ctypes.c_int),
    ctypes.c_int, ctypes.POINTER(ctypes.c_int)
]
lib.reduce_boundary.restype = None

BASE = '/content/drive/MyDrive/URP'
RIPS_DIR = os.path.join(BASE, 'Final_Vector', 'Sixpack_Rips')
A_VALS = [0.0, 0.01, 0.05, 0.09, 0.13, 0.17, 0.21, 0.25]
PARAMS = [(x1,x2,x3) for x1 in A_VALS for x2 in A_VALS for x3 in A_VALS]

In [ ]:
def compute_relative_barcode(A, B, max_edge=10):
    """Relative H*(K,L) barcode with C-accelerated column reduction."""
    total = np.concatenate([A, B]); a = len(A)
    rips = RipsComplex(points=total, max_edge_length=max_edge)
    st = rips.create_simplex_tree(max_dimension=2)
    pairs = [(tuple(sorted(s)), f) for s, f in st.get_filtration()]
    simplices, filt = [p[0] for p in pairs], [p[1] for p in pairs]
    del st, total; gc.collect()

    # K\L 분류
    idx_KmL = [i for i, s in enumerate(simplices) if any(v >= a for v in s)]
    set_KmL = set(idx_KmL)
    pos_map = {g: l for l, g in enumerate(idx_KmL)}
    sf_idx = {s: i for i, s in enumerate(simplices)}

    # Boundary matrix → flat array
    flat_list, offsets = [], [0]
    for g in idx_KmL:
        s = simplices[g]; bdry = []
        if len(s) > 1:
            for j in range(len(s)):
                fi = sf_idx.get(s[:j]+s[j+1:])
                if fi is not None and fi in set_KmL:
                    bdry.append(pos_map[fi])
        bdry.sort()
        flat_list.extend(bdry)
        offsets.append(len(flat_list))

    # C reduction
    m = len(idx_KmL)
    flat_arr = (ctypes.c_int * len(flat_list))(*flat_list)
    off_arr = (ctypes.c_int * len(offsets))(*offsets)
    low_arr = (ctypes.c_int * m)()
    lib.reduce_boundary(flat_arr, off_arr, m, low_arr)

    # Barcode 추출
    bars = defaultdict(list)
    for pos in range(m):
        if low_arr[pos] != -1:
            sigma = idx_KmL[low_arr[pos]]; tau = idx_KmL[pos]
            b, d = filt[sigma], filt[tau]
            if abs(b-d) > 1e-12:
                bars[len(simplices[sigma])-1].append((b, d))
    out = {}
    for p in [0, 1]:
        if bars[p]:
            arr = np.array(bars[p]); out[p] = arr[np.lexsort((arr[:,1],arr[:,0]))]
        else: out[p] = np.empty((0,2))
    return out

def compute_PIs(barcodes, max_eps=10, px_res=0.1, sigma=0.05):
    for k in barcodes:
        if len(barcodes[k]) == 0: barcodes[k] = np.zeros((0,2))
    vector = {}
    pi0 = PersistenceImager(pixel_size=px_res, birth_range=(0,1), pers_range=(0,max_eps))
    pi0.weight = weights.persistence; pi0.weight_params = {'n':1}
    pi0.kernel_params = {'sigma':[[sigma,0],[0,sigma]]}
    b0 = np.array(barcodes.get(0, np.zeros((0,2))))
    img0 = pi0.transform(b0, skew=False) if len(b0)>0 else np.zeros((int(1/px_res), int(max_eps/px_res)))
    vector[0] = np.mean(img0, axis=0)
    pi1 = PersistenceImager(pixel_size=px_res, birth_range=(0,max_eps), pers_range=(0,max_eps/2))
    pi1.weight = weights.persistence; pi1.weight_params = {'n':1}
    pi1.kernel_params = {'sigma':[[sigma,0],[0,sigma]]}
    b1 = np.array(barcodes.get(1, np.zeros((0,2))))
    img1 = pi1.transform(b1, skew=True) if len(b1)>0 else np.zeros((int(max_eps/px_res), int(max_eps/2/px_res)))
    vector[1] = img1.flatten()
    return vector

print('Functions defined (C-accelerated)')

## 3. 전체 512개 실행

In [ ]:
def ram(): return psutil.Process(os.getpid()).memory_info().rss/1024/1024

updated, skipped, errors = 0, 0, 0
t_total = time.time()

for idx in range(1, 513):
    rips_path = os.path.join(RIPS_DIR, f'Sixpack_Rips_{idx}.npz')
    if not os.path.exists(rips_path):
        skipped += 1; continue

    p = PARAMS[idx-1]
    pos_f = f'{BASE}/ParamSweep_{idx}_Output/Pos_{p[0]:.2f}_{p[1]:.2f}_{p[2]:.2f}.dat'
    typ_f = f'{BASE}/ParamSweep_{idx}_Output/Types_{p[0]:.2f}_{p[1]:.2f}_{p[2]:.2f}.dat'
    if not os.path.exists(pos_f):
        skipped += 1; continue

    print(f'[{idx:>3d}] ', end='')
    t0 = time.time()
    try:
        types = np.loadtxt(typ_f, dtype=int)
        pos = np.loadtxt(pos_f, delimiter=',')
        A, B = pos[types==1], pos[types==2]
        del types, pos

        rel_A2B = compute_PIs(compute_relative_barcode(A, B, 10), max_eps=10)
        gc.collect()
        rel_B2A = compute_PIs(compute_relative_barcode(B, A, 10), max_eps=10)
        del A, B; gc.collect()

        data = np.load(rips_path, allow_pickle=True)
        sp_A2B = data['arr_0'].item()
        sp_B2A = data['arr_1'].item()
        sp_A2B['relative'] = rel_A2B
        sp_B2A['relative'] = rel_B2A

        np.savez_compressed(rips_path, sp_A2B, sp_B2A)
        del data, sp_A2B, sp_B2A, rel_A2B, rel_B2A; gc.collect()
        updated += 1
        dt = time.time()-t0
        if updated % 20 == 0:
            elapsed = time.time()-t_total
            eta = elapsed/updated*(512-updated)/60
            print(f'OK ({dt:.1f}s) [{updated}/512, ETA {eta:.0f}min, RAM {ram():.0f}MB]')
        else:
            print(f'OK ({dt:.1f}s)')
    except Exception as e:
        errors += 1; print(f'ERR: {e}')

print(f'\nDone in {(time.time()-t_total)/60:.1f}min: updated={updated}, skipped={skipped}, errors={errors}')

## 4. 검증

In [ ]:
d = np.load(os.path.join(RIPS_DIR, 'Sixpack_Rips_1.npz'), allow_pickle=True)
sp = d['arr_0'].item()
for k in sorted(sp.keys()):
    v = sp[k]
    if isinstance(v, dict):
        dims = {dk: np.asarray(dv).shape for dk, dv in v.items()}
        print(f'{k:15s}: {dims}')
print('\n✓ 모든 키가 H0=(100,), H1=(5000,)이면 성공')